In [ ]:
%pip install -r requirements.txt

# Flight Delay Prediction System — Charlotte Douglas Intl (KCLT)

This notebook implements the pipeline described in the project README:
collect weather data -> store it in S3 -> preprocess -> train a model ->
predict delay probability -> monitor -> retrain.

**Important note on data:** the Aviation Weather Center API only gives us
*weather* (METAR observations). It has no idea whether a flight was delayed.
To train a real model we need a second dataset with actual flight delay
outcomes, joined to the weather at the time of departure. The standard free
source for that is the US DOT's **Bureau of Transportation Statistics (BTS)
On-Time Performance data**:
<https://www.transtats.bts.gov/DL_SelectFields.asp?gnoyr_VQ=FGJ> — filter to
`CLT` as the origin airport, download a month or two as CSV, and place it in
`data/flight_delays.csv`. The columns we use are `FL_DATE`, `DEP_TIME`,
`DEP_DELAY`, and `CANCELLED`.

In [10]:
# --- Setup & configuration -------------------------------------------------
import os
import io
import json
import time
from datetime import datetime, timedelta

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import boto3
from botocore.exceptions import NoCredentialsError, ClientError
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

STATION_ID = "KCLT"                 
S3_BUCKET = "uncc-dtsc-weatherapp-bucket"
AWS_REGION = "us-east-2"
RAW_WEATHER_KEY = "raw/metar_{ts}.json"       
MODEL_LOCAL_PATH = "models/delay_model.joblib"
DELAY_THRESHOLD_MIN = 15            # DOT's standard definition of a "delayed" flight

os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

## 1. `collect_weather_data()`

Pulls current (or recent) METAR observations for KCLT from the Aviation
Weather Center's public data API. No API key is required, but the service
does ask that you set a real `User-Agent` and stay under 1 request/minute
per thread — fine for a scheduled job pulling data every few minutes.

Docs: <https://aviationweather.gov/data/api/></br>
Endpoint used: `https://aviationweather.gov/api/data/metar`

In [11]:
def collect_weather_data(station_id: str = STATION_ID, hours_back: int = 2) -> pd.DataFrame:
    """Fetch recent METAR observations for a station from aviationweather.gov.

    Parameters
    ----------
    station_id : ICAO code, e.g. "KCLT"
    hours_back : how many hours of history to request (API allows up to 15 days,
                 but for a live pipeline you'd normally just want the last hour or two)

    Returns
    -------
    DataFrame with one row per observation and the raw + decoded METAR fields
    we care about for modeling.
    """
    url = "https://aviationweather.gov/api/data/metar"
    params = {
        "ids": station_id,
        "format": "json",
        "hours": hours_back,
    }
    headers = {"User-Agent": "rwci-flight-delay-coursework/1.0 (student project)"}

    resp = requests.get(url, params=params, headers=headers, timeout=15)
    resp.raise_for_status()

    if resp.status_code == 204 or not resp.text.strip():
        # Valid request, but nothing available right now
        return pd.DataFrame()

    records = resp.json()
    df = pd.json_normalize(records)

    # Keep + rename the fields that matter for delay prediction
    keep = {
        "icaoId": "station_id",
        "obsTime": "obs_time_epoch",
        "temp": "temp_c",
        "dewp": "dewpoint_c",
        "wspd": "wind_speed_kt",
        "wgst": "wind_gust_kt",
        "visib": "visibility_sm",
        "altim": "altimeter_hpa",
        "wxString": "weather_string",
        "rawOb": "raw_metar",
    }
    df = df.rename(columns=keep)
    df = df[[c for c in keep.values() if c in df.columns]]

    if "obs_time_epoch" in df.columns:
        df["obs_time"] = pd.to_datetime(df["obs_time_epoch"], unit="s", utc=True)

    return df


# Quick smoke test (safe to run without any AWS setup)
sample_weather = collect_weather_data(hours_back=3)
sample_weather.head()

,station_id,obs_time_epoch,temp_c,dewpoint_c,wind_speed_kt,visibility_sm,altimeter_hpa,raw_metar,obs_time
0,KCLT,1790394720,12.8,8.3,6,10+,1018.0,METAR KCLT 260352Z 33006KT 10SM FEW250 13/08 A...,2026-09-26 03:52:00+00:00
1,KCLT,1790391120,13.9,9.4,4,10+,1018.0,METAR KCLT 260252Z 33004KT 10SM FEW250 14/09 A...,2026-09-26 02:52:00+00:00
2,KCLT,1790387520,15.0,8.9,5,10+,1017.7,METAR KCLT 260152Z 34005KT 10SM FEW250 15/09 A...,2026-09-26 01:52:00+00:00


## 2. `store_data_s3()`

Uploads whatever we collected to the S3 bucket you create (see the setup
guide). Reads AWS credentials the normal `boto3` way — from environment
variables or `~/.aws/credentials` — **never hardcode keys in this file**.

If no bucket/credentials are configured yet, this falls back to saving a
local copy in `data/` so the rest of the notebook still works.

In [12]:
def store_data_s3(df: pd.DataFrame, bucket: str = S3_BUCKET,
                   key_pattern: str = RAW_WEATHER_KEY) -> str:
    """Upload a DataFrame of weather observations to S3 as JSON.

    Returns the S3 key (or local path, if S3 isn't reachable) it was written to.
    """
    if df.empty:
        print("Nothing to store — DataFrame is empty.")
        return ""

    ts = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    key = key_pattern.format(ts=ts)
    payload = df.to_json(orient="records", date_format="iso")

    try:
        s3 = boto3.client("s3", region_name=AWS_REGION)
        s3.put_object(Bucket=bucket, Key=key, Body=payload, ContentType="application/json")
        print(f"Uploaded {len(df)} rows to s3://{bucket}/{key}")
        return f"s3://{bucket}/{key}"
    except (NoCredentialsError, ClientError) as e:
        # Fallback so the notebook is still runnable before AWS is set up
        local_path = os.path.join("data", f"metar_{ts}.json")
        with open(local_path, "w") as f:
            f.write(payload)
        print(f"S3 upload skipped ({e}). Saved locally to {local_path} instead.")
        return local_path


def load_data_from_s3(bucket: str = S3_BUCKET, prefix: str = "raw/") -> pd.DataFrame:
    """Read back every raw weather file under a prefix and concatenate them."""
    try:
        s3 = boto3.client("s3", region_name=AWS_REGION)
        objs = s3.list_objects_v2(Bucket=bucket, Prefix=prefix).get("Contents", [])
        frames = []
        for obj in objs:
            body = s3.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read()
            frames.append(pd.read_json(io.BytesIO(body)))
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    except (NoCredentialsError, ClientError) as e:
        print(f"Could not read from S3 ({e}). Falling back to local data/ folder.")
        frames = [pd.read_json(os.path.join("data", f))
                  for f in os.listdir("data") if f.startswith("metar_")]
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# Try storing the sample we pulled above
store_data_s3(sample_weather)

C:\Users\evilr\AppData\Local\Temp\ipykernel_21496\1078102162.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


Uploaded 3 rows to s3://uncc-dtsc-weatherapp-bucket/raw/metar_20260926T044647Z.json


's3://uncc-dtsc-weatherapp-bucket/raw/metar_20260926T044647Z.json'

## 3. `preprocess_data()`

Joins the weather observations to actual flight outcomes (the BTS CSV
mentioned above) and builds the feature table the model trains on.

Each flight is matched to the *closest prior* METAR observation (weather
right before departure is what would have been "known" ahead of time).

In [13]:
def load_flight_delay_data(path: str = "data/flight_delays.csv") -> pd.DataFrame:
    """Load BTS On-Time Performance data (must be downloaded manually — see the
    note at the top of this notebook) for flights departing KCLT."""
    if not os.path.exists(path):
        print(f"No flight delay file found at {path}. "
              "Download one from the BTS On-Time Performance database first.")
        return pd.DataFrame()

    flights = pd.read_csv(path)
    flights = flights.rename(columns={
        "FL_DATE": "flight_date",
        "DEP_TIME": "dep_time",
        "DEP_DELAY": "dep_delay_min",
        "CANCELLED": "cancelled",
    })
    flights = flights.dropna(subset=["dep_time"])
    # Build a real timestamp out of FL_DATE + DEP_TIME (BTS gives DEP_TIME as HHMM)
    flights["dep_time"] = flights["dep_time"].astype(int).astype(str).str.zfill(4)
    flights["scheduled_dep_ts"] = pd.to_datetime(
        flights["flight_date"] + " " + flights["dep_time"].str[:2] + ":" + flights["dep_time"].str[2:],
        errors="coerce", utc=True,
    )
    flights["delayed"] = (flights["dep_delay_min"] >= DELAY_THRESHOLD_MIN).astype(int)
    return flights


def preprocess_data(weather_df: pd.DataFrame, flights_df: pd.DataFrame) -> pd.DataFrame:
    """Merge weather + flight outcomes and engineer model features."""
    if weather_df.empty or flights_df.empty:
        print("Need both weather and flight data to preprocess — returning empty frame.")
        return pd.DataFrame()

    weather_df = weather_df.sort_values("obs_time")
    flights_df = flights_df.sort_values("scheduled_dep_ts").dropna(subset=["scheduled_dep_ts"])

    merged = pd.merge_asof(
        flights_df, weather_df,
        left_on="scheduled_dep_ts", right_on="obs_time",
        direction="backward", tolerance=pd.Timedelta("2h"),
    )
    merged = merged.dropna(subset=["temp_c", "wind_speed_kt"])

    # Simple, interpretable features — this is an entry-level model, not a
    # research pipeline, so we keep the feature set small and explainable.
    merged["low_visibility"] = (merged["visibility_sm"].fillna(10) < 3).astype(int)
    merged["has_gusts"] = merged["wind_gust_kt"].notna().astype(int)
    merged["bad_weather_string"] = merged["weather_string"].notna().astype(int)

    feature_cols = [
        "temp_c", "dewpoint_c", "wind_speed_kt", "visibility_sm",
        "altimeter_hpa", "low_visibility", "has_gusts", "bad_weather_string",
    ]
    for c in feature_cols:
        if c not in merged.columns:
            merged[c] = 0
    merged[feature_cols] = merged[feature_cols].fillna(merged[feature_cols].median())

    return merged[feature_cols + ["delayed"]]


processed = preprocess_data(load_data_from_s3(), load_flight_delay_data())
processed.head()

No flight delay file found at data/flight_delays.csv. Download one from the BTS On-Time Performance database first.
Need both weather and flight data to preprocess — returning empty frame.


""


## 4. `train_model()`

A plain `RandomForestClassifier` — nothing fancy, which fits both the scope
of the assignment and keeps it easy to explain in the writeup. Shows basic
accuracy/precision/recall and one bare-bones feature-importance bar chart.

In [14]:
FEATURE_COLS = ["temp_c", "dewpoint_c", "wind_speed_kt", "visibility_sm",
                "altimeter_hpa", "low_visibility", "has_gusts", "bad_weather_string"]

def train_model(data: pd.DataFrame, feature_cols=FEATURE_COLS, save_path: str = MODEL_LOCAL_PATH):
    """Train/test split, fit a RandomForestClassifier, print basic metrics,
    and save the model to disk with joblib."""
    if data.empty:
        print("No processed data to train on yet.")
        return None, {}

    X = data[feature_cols]
    y = data["delayed"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y if y.nunique() > 1 else None
    )

    model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    metrics = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
    }
    print(metrics)

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    joblib.dump(model, save_path)

    # Bare-bones feature importance plot — no styling, just the numbers
    importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values()
    plt.figure(figsize=(6, 4))
    plt.barh(importances.index, importances.values)
    plt.xlabel("Feature importance")
    plt.title("Which weather features matter most")
    plt.tight_layout()
    plt.show()

    return model, metrics


model, train_metrics = train_model(processed)

No processed data to train on yet.


## 5. `predict_delay()`

Takes a fresh weather reading (e.g. straight out of `collect_weather_data()`)
and returns the predicted probability that a flight departing under those
conditions would be delayed.

In [15]:
def predict_delay(weather_row: dict, model=None, model_path: str = MODEL_LOCAL_PATH,
                   feature_cols=FEATURE_COLS) -> float:
    """Return the predicted probability (0-1) of a delay given current weather.

    `weather_row` should have the same keys used in preprocess_data's feature_cols,
    e.g. {"temp_c": 22, "dewpoint_c": 18, "wind_speed_kt": 12, "visibility_sm": 10,
          "altimeter_hpa": 1015, "low_visibility": 0, "has_gusts": 0, "bad_weather_string": 0}
    """
    if model is None:
        if not os.path.exists(model_path):
            raise FileNotFoundError("No trained model found — run train_model() first.")
        model = joblib.load(model_path)

    row = pd.DataFrame([{c: weather_row.get(c, 0) for c in feature_cols}])
    probability = model.predict_proba(row)[0][1]
    return float(probability)


# Example: normal weather vs. a rough thunderstorm day
calm_day = {"temp_c": 24, "dewpoint_c": 17, "wind_speed_kt": 8, "visibility_sm": 10,
            "altimeter_hpa": 1017, "low_visibility": 0, "has_gusts": 0, "bad_weather_string": 0}
stormy_day = {"temp_c": 26, "dewpoint_c": 24, "wind_speed_kt": 28, "visibility_sm": 2,
              "altimeter_hpa": 1002, "low_visibility": 1, "has_gusts": 1, "bad_weather_string": 1}

if model is not None:
    print("Calm day delay probability:  ", round(predict_delay(calm_day, model), 3))
    print("Stormy day delay probability:", round(predict_delay(stormy_day, model), 3))

## 6. `monitor_model()`

Checks the model's accuracy on new/incoming data and (optionally) publishes
that number to CloudWatch as a custom metric, matching the README's
"CloudWatch monitors errors and model performance" description. If no AWS
credentials are set up, it just prints the result — the monitoring logic
still works, it just skips the cloud part.

In [16]:
def monitor_model(model, X_new: pd.DataFrame, y_new: pd.Series,
                   push_to_cloudwatch: bool = True) -> dict:
    """Score the model on fresh data and report/alert on performance drops."""
    preds = model.predict(X_new)
    acc = accuracy_score(y_new, preds)
    result = {"accuracy": acc, "n_samples": len(y_new), "checked_at": datetime.utcnow().isoformat()}
    print(f"Monitoring check: accuracy={acc:.3f} on {len(y_new)} samples")

    if push_to_cloudwatch:
        try:
            cw = boto3.client("cloudwatch", region_name=AWS_REGION)
            cw.put_metric_data(
                Namespace="FlightDelayPrediction",
                MetricData=[{
                    "MetricName": "ModelAccuracy",
                    "Value": acc,
                    "Unit": "None",
                }],
            )
        except (NoCredentialsError, ClientError) as e:
            print(f"(CloudWatch metric not sent — {e})")

    return result


ACCURACY_ALERT_THRESHOLD = 0.70  # below this, retrain_model() kicks in

## 7. `retrain_model()`

Simple retraining loop: if `monitor_model()` reports accuracy below the
threshold, pull the latest data, retrain, and only replace the saved model
if the new one is actually better (so a bad retrain never overwrites a good
model).

In [17]:
def retrain_model(current_model, current_metrics: dict, new_data: pd.DataFrame,
                   feature_cols=FEATURE_COLS, save_path: str = MODEL_LOCAL_PATH):
    """Retrain on new_data and keep whichever model performs better."""
    if new_data.empty:
        print("No new data available to retrain on.")
        return current_model, current_metrics

    candidate_model, candidate_metrics = train_model(new_data, feature_cols, save_path="models/candidate.joblib")

    if candidate_metrics.get("accuracy", 0) > current_metrics.get("accuracy", 0):
        os.replace("models/candidate.joblib", save_path)
        print("New model performs better — replacing the deployed model.")
        return candidate_model, candidate_metrics
    else:
        print("New model did not beat the current one — keeping the existing model.")
        return current_model, current_metrics

## Putting it together

A minimal end-to-end run. In production this would be four separate
scheduled jobs (collect -> preprocess/train -> predict -> monitor), but here
it's one linear pass so you can see the whole pipeline at once.

In [18]:
# 1. Collect + store live weather
live_weather = collect_weather_data()
store_data_s3(live_weather)

# 2. Preprocess using everything stored so far + the BTS flight delay file
history = preprocess_data(load_data_from_s3(), load_flight_delay_data())

# 3. Train (only meaningful once data/flight_delays.csv actually exists)
model, metrics = train_model(history)

# 4. Predict for the current weather
if model is not None and not live_weather.empty:
    latest = live_weather.iloc[-1].to_dict()
    print("Current delay probability:", round(predict_delay(latest, model), 3))

# 5. Monitor + retrain if needed
if model is not None:
    check = monitor_model(model, history[FEATURE_COLS], history["delayed"])
    if check["accuracy"] < ACCURACY_ALERT_THRESHOLD:
        model, metrics = retrain_model(model, metrics, history)

C:\Users\evilr\AppData\Local\Temp\ipykernel_21496\1078102162.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


Uploaded 2 rows to s3://uncc-dtsc-weatherapp-bucket/raw/metar_20260926T044649Z.json
No flight delay file found at data/flight_delays.csv. Download one from the BTS On-Time Performance database first.
Need both weather and flight data to preprocess — returning empty frame.
No processed data to train on yet.
